# YelpZip EDA

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
})

## 1. 데이터 로드 & 필수 전처리 (라벨 변환)

In [16]:
df = pd.read_csv('yelpzip.csv', index_col=0)

# ✅ 필수 전처리: 라벨 변환 (-1 → 1 사기, 1 → 0 정상)
df['label'] = df['label'].map({-1: 1, 1: 0})

print(f"Total rows: {len(df):,}")
print(f"\n라벨 분포:")
print(df['label'].value_counts().rename({0: '정상(0)', 1: '사기(1)'}))
print(f"\n컬럼: {df.columns.tolist()}")
df.head(3)

Total rows: 608,458

라벨 분포:
label
정상(0)    528019
사기(1)     80439
Name: count, dtype: int64

컬럼: ['user_id', 'prod_id', 'rating', 'label', 'date', 'text', 'tag']


,user_id,prod_id,rating,label,date,text,tag
0,5044,0,1.0,1,2014-11-16,"Drinks were bad, the hot chocolate was watered...",fake
1,5045,0,1.0,1,2014-09-08,This was the worst experience I've ever had a ...,fake
2,5046,0,3.0,1,2013-10-06,This is located on the site of the old Spruce ...,fake


## 2. 기본 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (1) Label distribution pie chart
label_counts = df['label'].value_counts().sort_index()
wedges, texts, autotexts = axes[0].pie(
    label_counts,
    labels=['Normal(0)', 'Spam(1)'],
    autopct='%1.1f%%',
    colors=['#4C72B0', '#DD8452'],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontweight('bold')
axes[0].set_title('Label Distribution', fontweight='bold', pad=12)
axes[0].axis('off')

# (2) Rating distribution (overall)
rating_counts = df['rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, color='#4C72B0',
            edgecolor='white', linewidth=0.8, width=0.6)
axes[1].set_title('Rating Distribution (Overall)', fontweight='bold', pad=12)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Review Count')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].set_xticks([1, 2, 3, 4, 5])

# (3) Rating distribution by label
real_rating = df[df['label'] == 0]['rating'].value_counts().sort_index()
fake_rating = df[df['label'] == 1]['rating'].value_counts().sort_index()
x = [1, 2, 3, 4, 5]
width = 0.35
axes[2].bar([i - width/2 for i in x], real_rating.values, width, label='Normal',
            color='#4C72B0', edgecolor='white', linewidth=0.8)
axes[2].bar([i + width/2 for i in x], fake_rating.values, width, label='Spam',
            color='#DD8452', edgecolor='white', linewidth=0.8)
axes[2].set_title('Rating: Normal vs Spam', fontweight='bold', pad=12)
axes[2].set_xlabel('Rating')
axes[2].set_ylabel('Review Count')
axes[2].legend(frameon=False)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[2].set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.savefig('dist_basic.png', bbox_inches='tight')
plt.show()

## 3. 사용자 / 상품 활동량 분포

In [ ]:
user_review_cnt = df.groupby('user_id').size()
prod_review_cnt = df.groupby('prod_id').size()
user_spam_ratio = df.groupby('user_id')['label'].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(user_review_cnt, bins=50, color='#4C72B0', edgecolor='white', log=True)
axes[0].set_title('Reviews per User (log)', fontweight='bold', pad=12)
axes[0].set_xlabel('Review Count')
axes[0].set_ylabel('User Count (log)')

axes[1].hist(prod_review_cnt, bins=50, color='#55A868', edgecolor='white', log=True)
axes[1].set_title('Reviews per Product (log)', fontweight='bold', pad=12)
axes[1].set_xlabel('Review Count')
axes[1].set_ylabel('Product Count (log)')

axes[2].hist(user_spam_ratio, bins=20, color='#DD8452', edgecolor='white')
axes[2].set_title('Spam Ratio per User', fontweight='bold', pad=12)
axes[2].set_xlabel('Spam Ratio (0=all normal, 1=all spam)')
axes[2].set_ylabel('User Count')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('dist_user_prod.png', bbox_inches='tight')
plt.show()

print(f"Total users: {len(user_review_cnt):,}")
print(f"Total products: {len(prod_review_cnt):,}")
print(f"Reviews/user — median: {user_review_cnt.median():.0f}, max: {user_review_cnt.max():,}")
print(f"Reviews/product — median: {prod_review_cnt.median():.0f}, max: {prod_review_cnt.max():,}")

## 4. 날짜 기반 분포

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M')

monthly = df.groupby(['year_month', 'label']).size().unstack(fill_value=0)
monthly.index = monthly.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(monthly.index, monthly[0], label='Normal', color='#4C72B0', linewidth=1.5)
axes[0].plot(monthly.index, monthly[1], label='Spam', color='#DD8452', linewidth=1.5)
axes[0].set_title('Monthly Review Count (Normal vs Spam)', fontweight='bold', pad=12)
axes[0].set_ylabel('Review Count')
axes[0].legend(frameon=False)
tick_step = max(1, len(monthly) // 20)
axes[0].set_xticks(range(0, len(monthly), tick_step))
axes[0].set_xticklabels(monthly.index[::tick_step], rotation=45, ha='right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

monthly['spam_ratio'] = monthly[1] / (monthly[0] + monthly[1])
axes[1].bar(range(len(monthly)), monthly['spam_ratio'], color='#DD8452', edgecolor='white', alpha=0.8)
axes[1].set_title('Monthly Spam Ratio', fontweight='bold', pad=12)
axes[1].set_ylabel('Spam Ratio')
axes[1].set_xticks(range(0, len(monthly), tick_step))
axes[1].set_xticklabels(monthly.index[::tick_step], rotation=45, ha='right')
axes[1].axhline(df['label'].mean(), color='black', linestyle='--', linewidth=1,
                label=f'Overall avg ({df["label"].mean():.2%})')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('dist_time.png', bbox_inches='tight')
plt.show()

## 5. t-SNE — 계층 샘플링(정상 3000 + 사기 3000)으로 진행

In [20]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import numpy as np

df['user_review_cnt'] = df['user_id'].map(user_review_cnt)
df['prod_review_cnt'] = df['prod_id'].map(prod_review_cnt)
df['user_spam_ratio'] = df['user_id'].map(user_spam_ratio)
df['text_len'] = df['text'].str.len()
df['month'] = df['date'].dt.month

sample_real = df[df['label'] == 0].sample(3000, random_state=42)
sample_fake = df[df['label'] == 1].sample(3000, random_state=42)
sample = pd.concat([sample_real, sample_fake]).reset_index(drop=True)

features = ['rating', 'user_review_cnt', 'prod_review_cnt', 'user_spam_ratio', 'text_len', 'month']
X = StandardScaler().fit_transform(sample[features])
y = sample['label'].values

print("t-SNE 실행 중... (1~2분 소요)")
tsne = TSNE(n_components=2, perplexity=40, random_state=42, max_iter=1000)
X_2d = tsne.fit_transform(X)
print("완료!")

t-SNE 실행 중... (1~2분 소요)
완료!


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

colors = {0: '#4C72B0', 1: '#DD8452'}

for label_val, label_name in [(0, 'Normal'), (1, 'Spam')]:
    mask = y == label_val
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=colors[label_val],
        label=label_name,
        alpha=0.5, s=15, edgecolors='none'
    )

ax.set_title('t-SNE: Normal vs Spam Reviews\n(rating, user activity, product reviews, spam ratio, text length, month)',
             fontweight='bold', pad=12)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(markerscale=2, frameon=False)
plt.tight_layout()
plt.savefig('tsne.png', bbox_inches='tight')
plt.show()